In [0]:
# ── Silver: dim_players SCD Type 2 ──────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType
from delta.tables import DeltaTable

bronze = spark.table("football_catalog.bronze.players")

incoming = (bronze
    .select(
        F.col("player_id").cast(IntegerType()),
        F.trim(F.col("name")).alias("player_name"),
        F.col("current_club_id").cast(IntegerType()),
        F.trim(F.col("position")).alias("position"),
        F.trim(F.col("country_of_birth")).alias("nationality"),
        F.to_date(F.col("date_of_birth")).alias("date_of_birth"),
        F.col("market_value_in_eur").cast("double")
    )
    .filter(F.col("player_id").isNotNull())
    .dropDuplicates(["player_id"])
    .withColumn("valid_from",  F.current_date())
    .withColumn("valid_to",    F.lit("9999-12-31").cast(DateType()))
    .withColumn("is_current",  F.lit(True))
)

target = "football_catalog.silver.dim_players"

if not spark.catalog.tableExists(target):
    incoming.write.format("delta").saveAsTable(target)
else:
    dt = DeltaTable.forName(spark, target)
    
    # 1. FIX: Wrap the entire chained merge operation in parentheses
    (dt.alias("t").merge(
        incoming.alias("s"),
        "t.player_id = s.player_id AND t.is_current = True"
    )
    .whenMatchedUpdate(
        condition="t.current_club_id != s.current_club_id",
        # Lowercase "false" is safer for SQL boolean expressions
        set={"valid_to": "current_date()", "is_current": "false"}
    ).execute())
    
    # 2. FIX: Filter 'incoming' so we only append NEW players, 
    # or players whose club just changed.
    existing_active = dt.toDF().filter(F.col("is_current") == True)
    
    records_to_insert = incoming.join(
        existing_active,
        on=[
            incoming.player_id == existing_active.player_id, 
            incoming.current_club_id == existing_active.current_club_id
        ],
        how="left_anti" # Keeps only rows in 'incoming' that DON'T match active records
    )
    
    # Safely append only the new/changed records
    records_to_insert.write.format("delta").mode("append").saveAsTable(target)

print("dim_players SCD2 done")
spark.sql(f"SELECT * FROM {target} WHERE is_current = True LIMIT 5").show()
